### ***The Goal Of This Notebook***
We have implemented three things in the bi_layer.py file which are:
**1. The Fair Price Estimator**
- This will estimate the price of the appliance as per user's demand.

**2. The Price Premium Detector**
- This will compare the Market Price(Actual) Vs Fair Price(Predicted)
- We will segregate the comparison in three segments:
- i. Great Deal -> Market Price < 10% of Fair Price.
- ii. Fair Deal -> Within 10% Range.
- iii. Overpriced -> Market Price is > 20% of Fair Price.

**3. Partial Input Inference**
- It allows user to Enter Only Partial Known Information of the appliance they want ti check.
- The unknown features are imputed automatically and this let's user see the predictions and Complete business Details for the similar product he/she is looking for.

**The Value Score (0-100)**
- We score the product from 0 to 100 based on the Higher returns and other feature density.
 

In [2]:
import sys
from pathlib import Path

root = Path.cwd()

while root != root.parent:
    if (root / "src").exists():
        break
    root = root.parent

sys.path.insert(0, str(root))

In [3]:
import pandas as pd
import numpy as np # Added for np.expm1
import warnings
warnings.filterwarnings("ignore")

# Updated to use a relative path - ensure this is present
df = pd.read_parquet("../../data/03.cleaned/multi_appliances_cleaned_engineered.parquet")


import importlib
import src.bi_layer
# Crucial: Reload the module to pick up all changes in bi_layer.py
importlib.reload(src.bi_layer)


<module 'src.bi_layer' from 'd:\\Study\\data_science\\Appliance Intelligence Price Tracker and Recommender\\src\\bi_layer.py'>

In [4]:
# test_bi_layer.py

import pandas as pd
import numpy as np
from src.bi_layer import (
    _load_model,
    _load_default_feature_values,
    predict_price, # Though not directly used in BI functions, good to have
    get_shap_explanation,
    get_pricing_verdict,
    get_fair_price,
    get_prediction_range,
    interpret_brand_premium,
    build_feature_dataframe # We can test this independently too
)

print("BI Layer module reloaded and functions imported.")
print("-" * 60)

# Load actual training data
x_df = pd.read_parquet(
    r"D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/X_train.parquet"
)

y_df = pd.read_parquet(
    r"D:/Study/data_science/Appliance Intelligence Price Tracker and Recommender/models/dataset/y_train.parquet"
)

BI Layer module reloaded and functions imported.
------------------------------------------------------------


In [5]:
# Create a sample COMPLETE feature dict from actual data
sample_row_idx = 10 # chossing a different index for variety
complete_test_features_dict = x_df.loc[sample_row_idx].to_dict()
observed_price_for_complete_test = float(np.expm1(y_df.loc[sample_row_idx, 'log_price']))

print(f"Sample complete features dictionary created from X_train index {sample_row_idx}.")
print(f"Number of features in complete_test_features_dict: {len(complete_test_features_dict)}")
print("=" * 60)

# --- Test 1: Full Input Scenario (using the updated get_fair_price) ---
print("\n--- Test Scenario 1: Full Input (using updated get_fair_price) ---")
full_input_result = get_fair_price(complete_test_features_dict)
full_input_verdict = get_pricing_verdict(
    observed_price=observed_price_for_complete_test,
    predicted_price=full_input_result['fair_price']
)
full_input_range = get_prediction_range(complete_test_features_dict)

print(f"Predicted Fair Price: ₹{full_input_result['fair_price']:,.0f}")
print(f"Actual Price By User: ₹{observed_price_for_complete_test:,.0f}")
print(f"Prediction Range    : ₹{full_input_range[0]:,.0f} - ₹{full_input_range[1]:,.0f}")
print(f"Brand Premium       : {full_input_result['brand_premium']}")
print(f"Verdict             : {full_input_verdict['verdict']}")

# UPDATED: Conditional printing for savings/overpayment
if full_input_verdict['consumer_advantage'] > 0:
    print(f"Consumer Savings    : ₹{full_input_verdict['consumer_advantage']:,.0f} (compared to fair price)")
elif full_input_verdict['consumer_disadvantage'] > 0:
    print(f"Potential Overpayment: ₹{full_input_verdict['consumer_disadvantage']:,.0f}")
else:
    print("Financial Impact    : No significant savings or overpayment relative to fair price.")

print(f"Top SHAP Features (Full Input):")
for feature, value in full_input_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")
print("-" * 60)

Sample complete features dictionary created from X_train index 10.
Number of features in complete_test_features_dict: 65

--- Test Scenario 1: Full Input (using updated get_fair_price) ---
Predicted Fair Price: ₹33,321
Actual Price By User: ₹23,990
Prediction Range    : ₹28,323 - ₹38,320
Brand Premium       : Brand effect could not be isolated from other factors.
Verdict             : Exceptional Value
Consumer Savings    : ₹9,331 (compared to fair price)
Top SHAP Features (Full Input):
  - remainder__wm_top_load: 0.12
  - remainder__capacity_above_avg: 0.05
  - remainder__ref_single_door: 0.04
  - remainder__capacity_rating: -0.04
  - remainder__wm_semi_automatic: 0.03
------------------------------------------------------------


In [6]:
x_df.info()

<class 'pandas.DataFrame'>
Index: 2296 entries, 2006 to 879
Data columns (total 65 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   rating                    2296 non-null   float64
 1   category                  2296 non-null   str    
 2   n_features                2296 non-null   int64  
 3   brand_name                2296 non-null   str    
 4   star_rating               2296 non-null   float64
 5   has_star_rating           2296 non-null   int64  
 6   has_inverter              2296 non-null   int64  
 7   has_wifi                  2296 non-null   int64  
 8   has_voice_control         2296 non-null   int64  
 9   has_app_control           2296 non-null   int64  
 10  smart_connectivity_score  2296 non-null   int64  
 11  ac_split                  2296 non-null   Int8   
 12  ac_window                 2296 non-null   Int8   
 13  ac_pm25_filter            2296 non-null   Int8   
 14  ac_hepa_filter        

In [7]:
# --- Test 2: Minimum Viable Input ---
print("\n--- Test Scenario 2: Minimum Viable Input ---")
# Given By User: Category, Brand, Capacity Value, Capacity Unit
partial_features_min = {
    'category': 'AC',
    'brand_name': 'LG'.lower(), # Pick a common brand from your data
    'capacity_value': 1.5,
    'capacity_ac_tons': 1 # if user gives Capacity unit, we convert it into 1 for whatever unit he says
}

print(f"Input features: {partial_features_min}")
min_input_result = get_fair_price(partial_features_min)
min_input_range = get_prediction_range(partial_features_min) # get_prediction_range also needs to be updated to use build_feature_dataframe

print(f"Predicted Fair Price: ₹{min_input_result['fair_price']:,.0f}")
print(f"Prediction Range    : ₹{min_input_range[0]:,.0f} - ₹{min_input_range[1]:,.0f}")
print(f"Brand Premium       : {min_input_result['brand_premium']}")
print(f"Top SHAP Features (Min Inputs):")
for feature, value in min_input_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")
print("-" * 60)


--- Test Scenario 2: Minimum Viable Input ---
Input features: {'category': 'AC', 'brand_name': 'lg', 'capacity_value': 1.5, 'capacity_ac_tons': 1}
Predicted Fair Price: ₹39,642
Prediction Range    : ₹33,696 - ₹45,588
Brand Premium       : About ₹2,489 of the estimated price is attributable to brand effects. Premium brands tend to command this adjustment.
Top SHAP Features (Min Inputs):
  - remainder__capacity_above_avg: 0.18
  - remainder__wm_top_load: 0.11
  - target_enc__brand_name: 0.08
  - remainder__wm_semi_automatic: 0.03
  - remainder__ref_single_door: 0.02
------------------------------------------------------------


In [14]:
# --- Test 3: Intermediate Inputs ---
print("\n--- Test Scenario 3: Intermediate Inputs (adding Energy Rating, Inverter) ---")
partial_features_intermediate = {
    'category': 'AC',
    'brand_name': 'LG'.lower(),
    'capacity_value': 1.5,
    'capacity_ac_tons': 1,
    'star_rating': 5,      # Added
    'has_inverter': 1,        # Added
    'ac_split': 1 # Added
}

print(f"Input features: {partial_features_intermediate}")
intermediate_input_result = get_fair_price(partial_features_intermediate)
intermediate_input_range = get_prediction_range(partial_features_intermediate) # get_prediction_range also needs to be updated

print(f"Predicted Fair Price: ₹{intermediate_input_result['fair_price']:,.0f}")
print(f"Prediction Range    : ₹{intermediate_input_range[0]:,.0f} - ₹{intermediate_input_range[1]:,.0f}")
print(f"Brand Premium       : {intermediate_input_result['brand_premium']}")
print(f"Top SHAP Features (Intermediate Inputs):")
for feature, value in intermediate_input_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")
print("-" * 60)

# Compare min vs intermediate
price_diff = intermediate_input_result['fair_price'] - min_input_result['fair_price']
print(f"\nComparison: Price changed by ₹{price_diff:,.0f} by adding more details.")
print("This indicates that providing more information refines the prediction.")
print("=" * 60)


--- Test Scenario 3: Intermediate Inputs (adding Energy Rating, Inverter) ---
Input features: {'category': 'AC', 'brand_name': 'lg', 'capacity_value': 1.5, 'capacity_ac_tons': 1, 'star_rating': 5, 'has_inverter': 1, 'ac_split': 1}
Predicted Fair Price: ₹44,859
Prediction Range    : ₹38,130 - ₹51,588
Brand Premium       : About ₹2,250 of the estimated price is attributable to brand effects. Premium brands tend to command this adjustment.
Top SHAP Features (Intermediate Inputs):
  - remainder__capacity_above_avg: 0.18
  - remainder__wm_top_load: 0.11
  - remainder__star_rating: 0.08
  - target_enc__brand_name: 0.08
  - remainder__has_inverter: 0.03
------------------------------------------------------------

Comparison: Price changed by ₹5,217 by adding more details.
This indicates that providing more information refines the prediction.


In [9]:
# --- Test 4: Impact of a Specific Default ---
print("\n--- Test Scenario 4: Impact of a Specific Default (missing 'brand_name') ---")
# Take the intermediate features and remove 'Inverter' to see the default effect
features_missing_inverter = partial_features_intermediate.copy()
# Safely remove 'Inverter' if it exists
features_missing_inverter.pop('brand_name', None)

print(f"Input features (missing Brand Name): {features_missing_inverter}")
missing_inverter_result = get_fair_price(features_missing_inverter)
missing_inverter_range = get_prediction_range(features_missing_inverter) # get_prediction_range also needs to be updated

print(f"Predicted Fair Price (Intermediate): ₹{intermediate_input_result['fair_price']:,.0f}")
print(f"Predicted Fair Price (Missing Brand Name): ₹{missing_inverter_result['fair_price']:,.0f}")
print(f"Top SHAP Features (Missing Brand Name):")
for feature, value in missing_inverter_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")

price_change_inverter = missing_inverter_result['fair_price'] - intermediate_input_result['fair_price']
print(f"\nComparison: Price changed by ₹{price_change_inverter:,.0f} when 'Brand Name' was defaulted.")
# If Inverter is typically a premium feature, defaulting it might lower the price.
print("This shows how defaults influence the prediction.")
print("=" * 60)


--- Test Scenario 4: Impact of a Specific Default (missing 'brand_name') ---
Input features (missing Brand Name): {'category': 'AC', 'capacity_value': 1.5, 'capacity_ac_ton': 1, 'star_rating': 5, 'has_inverter': 1, 'ac_split': 1}
Predicted Fair Price (Intermediate): ₹44,859
Predicted Fair Price (Missing Brand Name): ₹41,647
Top SHAP Features (Missing Brand Name):
  - remainder__capacity_above_avg: 0.19
  - remainder__wm_top_load: 0.12
  - remainder__star_rating: 0.07
  - remainder__has_inverter: 0.03
  - remainder__ref_single_door: 0.03

Comparison: Price changed by ₹-3,212 when 'Brand Name' was defaulted.
This shows how defaults influence the prediction.


In [10]:
# --- Test 5: Pricing Verdict Categories ---
print("\n--- Test Scenario 5: Pricing Verdict Categories ---")

# Let's use the 'min_input_result' as our base predicted price for these tests.
base_predicted_price = min_input_result['fair_price']

# Scenario A: Fairly Priced (within +/- 10%)
observed_price_A = base_predicted_price * 1.05 # 5% higher
verdict_A = get_pricing_verdict(observed_price=observed_price_A, predicted_price=base_predicted_price)
print(f"Predicted: ₹{base_predicted_price:,.0f}, Observed: ₹{observed_price_A:,.0f} -> Verdict: {verdict_A['verdict']}")
if verdict_A['consumer_advantage'] > 0:
    print(f"  Consumer Savings: ₹{verdict_A['consumer_advantage']:,.0f}")
elif verdict_A['consumer_disadvantage'] > 0:
    print(f"  Potential Overpayment: ₹{verdict_A['consumer_disadvantage']:,.0f}")

# Scenario B: Slightly Overpriced (10-20% higher)
observed_price_B = base_predicted_price * 1.15 # 15% higher
verdict_B = get_pricing_verdict(observed_price=observed_price_B, predicted_price=base_predicted_price)
print(f"Predicted: ₹{base_predicted_price:,.0f}, Observed: ₹{observed_price_B:,.0f} -> Verdict: {verdict_B['verdict']}")
if verdict_B['consumer_advantage'] > 0:
    print(f"  Consumer Savings: ₹{verdict_B['consumer_advantage']:,.0f}")
elif verdict_B['consumer_disadvantage'] > 0:
    print(f"  Potential Overpayment: ₹{verdict_B['consumer_disadvantage']:,.0f}")

# Scenario C: Overpriced (>20% higher)
observed_price_C = base_predicted_price * 1.25 # 25% higher
verdict_C = get_pricing_verdict(observed_price=observed_price_C, predicted_price=base_predicted_price)
print(f"Predicted: ₹{base_predicted_price:,.0f}, Observed: ₹{observed_price_C:,.0f} -> Verdict: {verdict_C['verdict']}")
if verdict_C['consumer_advantage'] > 0:
    print(f"  Consumer Savings: ₹{verdict_C['consumer_advantage']:,.0f}")
elif verdict_C['consumer_disadvantage'] > 0:
    print(f"  Potential Overpayment: ₹{verdict_C['consumer_disadvantage']:,.0f}")

# Scenario D: Good Deal (-10% to -20% lower)
observed_price_D = base_predicted_price * 0.85 # 15% lower
verdict_D = get_pricing_verdict(observed_price=observed_price_D, predicted_price=base_predicted_price)
print(f"Predicted: ₹{base_predicted_price:,.0f}, Observed: ₹{observed_price_D:,.0f} -> Verdict: {verdict_D['verdict']}")
if verdict_D['consumer_advantage'] > 0:
    print(f"  Consumer Savings: ₹{verdict_D['consumer_advantage']:,.0f}")
elif verdict_D['consumer_disadvantage'] > 0:
    print(f"  Potential Overpayment: ₹{verdict_D['consumer_disadvantage']:,.0f}")

# Scenario E: Exceptional Value (< -20% lower)
observed_price_E = base_predicted_price * 0.75 # 25% lower
verdict_E = get_pricing_verdict(observed_price=observed_price_E, predicted_price=base_predicted_price)
print(f"Predicted: ₹{base_predicted_price:,.0f}, Observed: ₹{observed_price_E:,.0f} -> Verdict: {verdict_E['verdict']}")
if verdict_E['consumer_advantage'] > 0:
    print(f"  Consumer Savings: ₹{verdict_E['consumer_advantage']:,.0f}")
elif verdict_E['consumer_disadvantage'] > 0:
    print(f"  Potential Overpayment: ₹{verdict_E['consumer_disadvantage']:,.0f}")

print("=" * 60)


--- Test Scenario 5: Pricing Verdict Categories ---
Predicted: ₹39,642, Observed: ₹41,624 -> Verdict: Fairly Priced
  Potential Overpayment: ₹1,982
Predicted: ₹39,642, Observed: ₹45,588 -> Verdict: Slightly Overpriced
  Potential Overpayment: ₹5,946
Predicted: ₹39,642, Observed: ₹49,552 -> Verdict: Overpriced
  Potential Overpayment: ₹9,910
Predicted: ₹39,642, Observed: ₹33,696 -> Verdict: Good Deal
  Consumer Savings: ₹5,946
Predicted: ₹39,642, Observed: ₹29,731 -> Verdict: Exceptional Value
  Consumer Savings: ₹9,910


In [12]:
# --- Test 6: Cross-Category SHAP and Brand Premium ---
print("\n--- Test Scenario 6: Cross-Category SHAP and Brand Premium ---")

# Refrigerator Example
fridge_features = {
    'category': 'Refrigerator',
    'brand_name': 'Samsung'.lower(), # Common brand
    'capacity_value': 250,
    'capacity_ref_litres': 1,
    'star_rating': 3,
    'has_inverter': 1,
    'ref_double_door': 1,
    'ref_frost_free': 1 # Category-specific feature
}
print(f"\n--- Refrigerator: {fridge_features['brand_name']} {fridge_features['capacity_value']}L {fridge_features['category']} ---")
fridge_result = get_fair_price(fridge_features)
print(f"Predicted Fair Price: ₹{fridge_result['fair_price']:,.0f}")
print(f"Brand Premium       : {fridge_result['brand_premium']}")
print(f"Top SHAP Features:")
for feature, value in fridge_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")

# Washing Machine Example
wm_features = {
    'category': 'WashingMachine',
    'brand_name': 'Bosch'.lower(), # Common brand
    'capacity_value': 7,
    'capacity_wm_kg': 1,
    'wm_front_load': 1,
    'has_inverter': True
}
print(f"\n--- Washing Machine: {wm_features['brand_name']} {wm_features['capacity_value']}KG {wm_features['category']} ---")
wm_result = get_fair_price(wm_features)
print(f"Predicted Fair Price: ₹{wm_result['fair_price']:,.0f}")
print(f"Brand Premium       : {wm_result['brand_premium']}")
print(f"Top SHAP Features:")
for feature, value in wm_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")

print("=" * 60)


--- Test Scenario 6: Cross-Category SHAP and Brand Premium ---

--- Refrigerator: samsung 250L Refrigerator ---
Predicted Fair Price: ₹41,017
Brand Premium       : About ₹1,809 of the estimated price is attributable to brand effects. Premium brands tend to command this adjustment.
Top SHAP Features:
  - remainder__capacity_above_avg: 0.18
  - remainder__wm_top_load: 0.11
  - target_enc__brand_name: 0.06
  - remainder__wm_semi_automatic: 0.03
  - remainder__ref_single_door: 0.03

--- Washing Machine: bosch 7KG WashingMachine ---
Predicted Fair Price: ₹39,302
Brand Premium       : About ₹1,889 of the estimated price is attributable to brand effects. Premium brands tend to command this adjustment.
Top SHAP Features:
  - remainder__capacity_above_avg: 0.18
  - remainder__wm_top_load: 0.11
  - target_enc__brand_name: 0.06
  - remainder__wm_semi_automatic: 0.03
  - remainder__ref_single_door: 0.03


In [13]:
# --- Test 7: Edge Case - Unknown Brand (to see default behavior) ---
print("\n--- Test Scenario 7: Edge Case - Unknown Brand (will use default brand) ---")
unknown_brand_features = {
    'category': 'AC',
    'brand_name': 'NonExistentBrand123', # This brand is likely not in your training data
    'capacity_value': 1.0,
    'capacity_ac_ton': 'Ton'
}
print(f"Input features: {unknown_brand_features}")
unknown_brand_result = get_fair_price(unknown_brand_features)
print(f"Predicted Fair Price: ₹{unknown_brand_result['fair_price']:,.0f}")
print(f"Brand Premium       : {unknown_brand_result['brand_premium']}")
print(f"Top SHAP Features (Unknown Brand):")
for feature, value in unknown_brand_result['explanation']['top_features'][:5]:
    print(f"  - {feature}: {value:.2f}")
print("-" * 60)


--- Test Scenario 7: Edge Case - Unknown Brand (will use default brand) ---
Input features: {'category': 'AC', 'brand_name': 'NonExistentBrand123', 'capacity_value': 1.0, 'capacity_ac_ton': 'Ton'}
Predicted Fair Price: ₹36,471
Brand Premium       : Brand effect could not be isolated from other factors.
Top SHAP Features (Unknown Brand):
  - remainder__capacity_above_avg: 0.18
  - remainder__wm_top_load: 0.11
  - remainder__has_inverter: 0.03
  - remainder__ref_single_door: 0.03
  - remainder__wm_semi_automatic: 0.03
------------------------------------------------------------
